# Phase 9 — Evals & Observability

**Goal**: How do you know your agent system is actually *good*? Not by eyeballing one answer — by running an **eval**. An eval runs the agent against a **golden set** (a fixed list of questions with known-correct answers), scores every answer, and reports aggregate metrics: **pass rate, latency, cost**.

This is **eval-driven development** — the discipline that turns "it seemed to work" into "it passes 5/5 at $0.01/query, p50 4s." Every `Trace` you've saved since Phase 1 was building toward this.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Golden set** | A fixed set of test questions + their correct answers. Your benchmark. |
| **Scoring** | Checking one answer against its expected facts. Pass / fail. |
| **The eval loop** | Run every golden case, score each, collect the results. |
| **Aggregate metrics** | Pass rate, average latency, total cost — the system's report card. |
| **Regression safety** | Re-run the eval after any change; if the pass rate drops, you broke something. |

## Acceptance criteria
1. The eval runs the agent over **every** golden case
2. Each case is scored and its `Trace` saved to `traces/traces.jsonl`
3. The aggregate **pass rate is at least 80%**

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
from orchestrator.agents import run_agent
# Phase 9 adds the evals module:
from orchestrator.evals import load_golden_set, score_answer, summarize, GoldenCase, EvalRow
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")
print(f"Dev model: {DEV_MODEL}")

## 2. The golden set

`tests/golden.json` is the **benchmark** — a fixed list of questions, each with the facts a correct answer must contain (`expected_substrings`). It was computed from the real data with pandas, so we *know* the right answers.

`load_golden_set()` loads it into `GoldenCase` objects.

In [ ]:
golden_cases = load_golden_set("tests/golden.json")

print(f"Golden set: {len(golden_cases)} cases\n")
for case in golden_cases:
    print(f"[{case.case_id}]")
    print(f"  Q: {case.question}")
    print(f"  expected: {case.expected_substrings}")
    print()

## 3. Rebuild the retail tool + DataAgent (recap)

Same `query_retail` tool + DataAgent. Just run it.

In [ ]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Arguments: year (optional), country (optional - "
        "OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2011"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])

data_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer using the query_retail tool "
        "and present the result as a markdown table. Only filter by country "
        "if the question names a country. Never invent numbers."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] Retail tool + DataAgent ready")

## 4. Scoring — how one answer is judged

`score_answer(answer, expected_substrings)` returns the expected facts that are **missing** from an answer. An empty list `[]` means every fact is present — a **pass**.

Run the cell to see scoring on a sample.

In [ ]:
sample_answer = "Top countries: United Kingdom, then Germany."
sample_expected = ["United Kingdom", "Germany", "France"]

missing = score_answer(sample_answer, sample_expected)
print(f"answer:   {sample_answer}")
print(f"expected: {sample_expected}")
print(f"missing:  {missing}")
print(f"-> {'PASS' if missing == [] else 'FAIL'}  (France is absent, so this sample fails)")

## 5. The eval loop

`run_eval()` runs the agent on **every** golden case, scores each answer, saves a `Trace`, and collects an `EvalRow`. Most of it is given — you write the **scoring** step.

**TODO 1**: score each case.

In [ ]:
async def run_eval(golden_cases, data_options, model):
    """Run the agent on every golden case; return a list of EvalRow results."""
    rows = []
    for case in golden_cases:
        trace = Trace(model=model, question=case.question)
        with Timer() as t:
            run = await run_agent("DataAgent", case.question, data_options)
        trace.input_tokens        = run.input_tokens
        trace.output_tokens       = run.output_tokens
        trace.cached_input_tokens = run.cached_input_tokens
        trace.n_tool_calls        = run.n_tool_calls
        trace.latency_ms          = t.elapsed_ms
        trace.answer              = run.answer
        trace.compute_cost()

        # ----- TODO 1: score this case --------------------------------------
        missing = score_answer(run.answer, case.expected_substrings)
        passed = (missing == [])
        # --------------------------------------------------------------------

        trace.passed = passed
        trace.append_jsonl()
        rows.append(EvalRow(
            case_id=case.case_id,
            passed=passed,
            missing=missing,
            latency_ms=trace.latency_ms,
            cost_usd=trace.cost_usd,
        ))
        print(f"  {case.case_id:34s} {'PASS' if passed else 'FAIL'}")
    return rows

print("[OK] run_eval defined")

## 6. Run the full eval

This runs the agent once per golden case — so it makes several LLM calls and takes a minute or two. Watch each case report PASS / FAIL as it finishes.

In [ ]:
print("Running eval over the golden set...\n")
with Timer() as total_timer:
    rows = await run_eval(golden_cases, data_options, DEV_MODEL)
print(f"\nEval finished in {total_timer.elapsed_ms / 1000:.1f}s")

## 7. The results table

Turn the `EvalRow` results into a table and headline metrics — the system's report card.

In [ ]:
# Build a results table from the eval rows
table_data = []
for r in rows:
    table_data.append({
        "case_id": r.case_id,
        "passed": r.passed,
        "latency_ms": r.latency_ms,
        "cost_usd": r.cost_usd,
        "missing": r.missing,
    })
results_df = pd.DataFrame(table_data)
print(results_df.to_string(index=False))

# Headline metrics
metrics = summarize(rows)
print()
print("----- Headline metrics -----")
print(f"  cases:           {metrics['n_cases']}")
print(f"  passed:          {metrics['n_passed']}")
print(f"  pass rate:       {metrics['pass_rate']:.0%}")
print(f"  avg latency:     {metrics['avg_latency_ms']:.0f} ms")
print(f"  total cost:      ${metrics['total_cost_usd']:.4f}")

## 8. Verify (acceptance criteria)

In [ ]:
# ----- TODO 2: write the assertion -----
eval_passed = metrics["pass_rate"] >= 0.8
# ---------------------------------------

print(f"Pass rate: {metrics['pass_rate']:.0%}  ({metrics['n_passed']}/{metrics['n_cases']})")
print(f"Eval passed (>= 80%): {eval_passed}")
assert eval_passed, f"Eval pass rate {metrics['pass_rate']:.0%} is below the 80% bar"

# Phase 9 acceptance: every case ran and was scored
assert len(rows) == len(golden_cases), "Every golden case should have an EvalRow"
print("\n[OK] Acceptance criteria met - eval ran over the full golden set, pass rate above bar")

## 9. Quiz (~5 min — answer in a new markdown cell)

**TODO 3**: Answer in your own words.

1. **Why a golden set**: you could just ask the agent a question and read the answer. Why is a fixed golden set with known answers better for judging the system?

   Because *one* answer tells you about *one* moment, not about the system. A golden set makes performance **measurable and repeatable** — same questions, same scoring, every time, so a change in pass rate is a signal about the system, not about which question you happened to ask today. It also takes the human out of the loop: substring scoring is deterministic, so the same answer always gets the same grade.

2. **Pass rate**: this eval scores by substring matching (are the expected facts present?). Name one kind of *wrong* answer this scoring would still mark as PASS. How might you make the scoring stricter?

   Substring matching only checks **presence**, not **correctness in context**. An answer like "United Kingdom, EIRE, Netherlands, Germany, France did NOT make the top 5" contains every expected country but is the literal opposite of true — and it passes. Stricter options: parse the markdown table and check ranking + magnitude, regex for "Country: revenue" pairs and validate the numbers against ground truth, or layer an LLM-judge eval on top to catch logically wrong answers that happen to contain the right words.

3. **Regression safety**: suppose you change a system prompt and the pass rate drops from 100% to 60%. What does that tell you, and what would you do?

   Tells you the prompt change **broke** something — three cases that used to work no longer do. Workflow: diff the failing cases' answers before and after the change (the traces are right there in `traces.jsonl`); look for the common failure mode (wrong tool args? omitted rows? new formatting that hides a country name?); decide between rolling back the prompt or fixing the specific regression while keeping whatever the change was meant to improve. The point of evals is that you find this out in 90 seconds locally instead of from a user three weeks later.

4. **The metrics**: the eval reports pass rate, latency, and cost together. Why look at all three — when might you accept a *lower* pass rate, or a *higher* cost?

   They're a Pareto front, not a single number. Pass rate is quality; latency is UX; cost is unit economics. **Accept a lower pass rate** when you're choosing a model for a high-volume routing layer where 80% accuracy at $0.001 beats 95% at $0.05. **Accept a higher cost** when the user is an exec waiting on a quarterly review — better to spend $0.20 on Sonnet and be right than $0.02 on Haiku and be plausible-but-wrong. The job isn't to maximize any one metric; it's to pick the right point on the curve for the use case.

5. **The whole project**: across Phases 1-9 you built tools, sub-agents, reflection, planning, memory, RAG, an MCP server, human approval, guardrails, and evals. In 2-3 sentences: what is the single most important thing you learned?

   That an "agent" isn't a model — it's a *system of small, opinionated parts*: a tool, a prompt, a reviewer, a memory, a gate, a guardrail, an eval. The interesting engineering is in the **seams between those parts**, not inside any one of them. And without observability and evals running from Phase 1, none of it is real — you can't improve what you can't measure, and "it worked when I tried it" is not measurement.

## Phase 9 done when:
- [x] TODO 1 (scoring step in the eval loop) filled in
- [x] TODO 2 (the assertion) filled in
- [x] TODO 3 (quiz) answered
- [x] Cell 13 runs the full eval and every case reports PASS/FAIL
- [x] Cell 15 shows the results table + headline metrics
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has one new line per golden case

That's Phase 9 — and the last guided notebook. Ping me: we'll review, fill in the README's results table together, and talk through Phase 10 (the Streamlit capstone) and your Medium/LinkedIn writeup.